In [1]:
!git clone https://github.com/echanatwell/LLM_weight_quantization_triton.git

Cloning into 'LLM_weight_quantization_triton'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 69 (delta 31), reused 58 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 25.97 KiB | 6.49 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [2]:
!pip install bitsandbytes --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 26.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.2 MB/s eta 0:00:00:00:0100:01
ERROR: pip's de

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.ao.quantization import get_default_qconfig_mapping
from torch.quantization.quantize_fx import prepare_fx, convert_fx

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

import triton
import triton.language as tl

import bitsandbytes as bnb
from bitsandbytes import functional as bbF

import time
import os
os.chdir('/kaggle/working/LLM_weight_quantization_triton')
import gc
from collections import defaultdict
import numpy as np
from tqdm import tqdm

from CustomLayers import DummyLinear, QuantizedLinearGlobalTorch, GlobalQuantLinearTriton, QuantizedLinearRowwiseTorch
from benchmark.perplexity import measure_ppl

In [52]:
def change_linear_layer(model, new_layer, device):
    for layer in model.model.layers:
        layer.self_attn.q_proj = new_layer(layer.self_attn.q_proj, device)
        layer.self_attn.k_proj = new_layer(layer.self_attn.k_proj, device)
        layer.self_attn.v_proj = new_layer(layer.self_attn.v_proj, device)
        layer.self_attn.o_proj = new_layer(layer.self_attn.o_proj, device)

        layer.mlp.gate_proj = new_layer(layer.mlp.gate_proj, device)
        layer.mlp.up_proj = new_layer(layer.mlp.up_proj, device)
        layer.mlp.down_proj = new_layer(layer.mlp.down_proj, device)

    model.lm_head = new_layer(model.lm_head, device)
    torch.cuda.empty_cache()

    return model


def calc_model_size(model):
    param_mem = 0.
    buffer_mem = 0.
    for param in model.parameters():
        param_mem += param.nelement() * param.element_size()
    for buffer in model.buffers():
        buffer_mem += buffer.nelement() * buffer.element_size()

    return (param_mem + buffer_mem) / (2 ** 30)


def time_inference(model, tokenizer, device, text, max_length=50):

    inputs = tokenizer(text, return_tensors='pt').to(device)

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_length, 
                               pad_token_id=tokenizer.eos_token_id)
    end_time = time.time()

    inference_time = end_time - start_time
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return inference_time, generated_text


def verify_layer_accuracy(orig_model, quant_model, tokenizer, text='hello'):
    inputs = tokenizer(text, return_tensors='pt').to(orig_model.device)
    inputs_ = tokenizer(text, return_tensors='pt').to(quant_model.device)

    with torch.no_grad():
        orig_outputs = orig_model(**inputs, output_hidden_states=True)

    with torch.no_grad():
        quant_outputs = quant_model(**inputs_, output_hidden_states=True)

    for i, (orig_hidden, quant_hidden) in enumerate(zip(orig_outputs.hidden_states, quant_outputs.hidden_states)):
        print(orig_hidden.shape, quant_hidden.shape)
        mse = F.mse_loss(orig_hidden.cpu(), quant_hidden.cpu())
        cos_sim = F.cosine_similarity(orig_hidden.flatten().cpu(), quant_hidden.flatten().cpu(), dim=0)
        print(f'layer {i}: mse = {mse:.6f}, cosine_sim = {cos_sim:.6f}')


def calc_quant_error_metrics(real_tensor, quant_tensor):
    rmse = torch.sqrt(F.mse_loss(real_tensor, quant_tensor))
    mean_ape = 100.0 * torch.mean(torch.abs((torch.abs(quant_tensor) - torch.abs(real_tensor))) / torch.abs(real_tensor))
    median_ape = 100.0 * torch.median(torch.abs((torch.abs(quant_tensor) - torch.abs(real_tensor))) / torch.abs(real_tensor))
    return {'rmse': rmse.cpu().numpy(), 'mean_ape': mean_ape.cpu().numpy(), 'median_ape': median_ape.cpu().numpy()}


def deallocate_tensors_from_gpu(*tensors):
    for t in tensors:
        t.to('cpu')
        del t
    gc.collect()
    torch.cuda.empty_cache()


def check_mm(layer, q_state=None):
    results = []
    for seq_len in [128, 512, 2048]:
        run_time = 0
        n_iter = 1000
        for _ in tqdm(range(n_iter)):
            x = torch.randn(1, seq_len, 2048).cuda()
            s = time.time()
            if q_state:
                dq_layer = bbF.dequantize_nf4(layer, q_state)
                x @ dq_layer
            else:
                layer(x)
            e = time.time()
            run_time += e - s
        results.append(round(run_time / n_iter * 1e6, 3))

    print(f'128: {results[0]} us') # * 1e-6
    print(f'512: {results[1]} us')
    print(f'2048: {results[2]} us')

In [16]:
# raw_datasets = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test") # longer sequnces
raw_datasets = load_dataset('zhengxuanzenwu/wikitext-2-split-128', split='test')

README.md:   0%|          | 0.00/161 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json:   0%|          | 0.00/809 [00:00<?, ?B/s]

data/train-00000-of-00001-4cf8ebea3d0e92(…):   0%|          | 0.00/7.70M [00:00<?, ?B/s]

data/validation-00000-of-00001-d5f9cad95(…):   0%|          | 0.00/888k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67922 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8192 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8192 [00:00<?, ? examples/s]

In [17]:
prompts = [x['text'] for x in raw_datasets if len(x['text']) > 0]
print('Number of sequences:', len(prompts))

Number of sequences: 8192


In [ ]:
model_id = 'unsloth/Llama-3.2-1B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map='cuda:0')

In [ ]:
# del model
# del custom_model
# torch.cuda.empty_cache()

In [ ]:
custom_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32, device_map='cuda:0')
custom_model = change_linear_layer(custom_model, GlobalQuantLinearTriton, custom_model.device)

In [ ]:
print(model.model.layers[0].self_attn.q_proj.weight.dtype)
(custom_model.model.layers[0].self_attn.q_proj.weight.dtype)

In [10]:
orig_ppl, orig_time = measure_ppl(prompts, model, tokenizer)

100%|██████████| 8192/8192 [08:13<00:00, 16.58it/s]



Perplexity: 345.0570
Mean time per sample: 0.060 s


In [11]:
custom_ppl, custom_time = measure_ppl(prompts, custom_model, tokenizer)

100%|██████████| 8192/8192 [07:48<00:00, 17.48it/s]


Perplexity: 345.0570
Mean time per sample: 0.057 s


In [14]:
# life could be dream
#
#
# def bench_matmul(seq_lens, linl_shapes, seed=59, num_iter=5, device='cuda:0', **kwargs):
#     torch.manual_seed(seed)
#     for seql in seq_lens:
#         for linls in linl_shapes:
#             stats = defaultdict(list)
#             N, IN, OUT = seql, linls[0], linls[1]
            
#             for _ in range(num_iter):
#                 x = torch.randn(N, IN, dtype=torch.float32, requires_grad=False).to(device)
#                 layer = torch.empty(IN, OUT, dtype=torch.float32, requires_grad=False).to(device)
#                 nn.init.kaiming_uniform_(layer, mode='fan_in', nonlinearity='relu')
#                 match kwargs:
#                     case {'bnb_config': config} if isinstance(config, BitsAndBytesConfig):
#                         q_layer, q_state = bbF.quantize_nf4(layer)
#                         tr0 = time.time()
#                         out_real = x @ layer
#                         tr1 = time.time()
#                         tq0 = time.time()
#                         dq_layer = bbF.dequantize_nf4(q_layer, q_state)
#                         out_quant = x @ dq_layer
#                         tq1 = time.time()
#                         deallocate_tensors_from_gpu(x, layer, dq_layer, q_layer, out_quant, out_real)
#                     case {'torch_global': True}:
#                         layer_torch = nn.Linear(IN, OUT, bias=False, dtype=torch.float32).to(device)
#                         with torch.no_grad():
#                             layer_torch.weight.copy_(layer)
#                         q_layer = QuantizedLinearGlobalTorch(layer_torch, device)
                
#                         tr0 = time.time()
#                         out_real = x @ layer
#                         tr1 = time.time()
#                         tq0 = time.time()
#                         out_quant = q_layer(x)
#                         tq1 = time.time()
#                         deallocate_tensors_from_gpu(x, layer, layer_torch, q_layer, out_quant, out_real)
#                     case {'torch_rowwise': True}:
#                         layer = nn.Linear(IN, OUT, bias=False, dtype=torch.float32).to(device)
#                         with torch.no_grad():
#                             layer_torch.weight.copy_(layer)
#                         q_layer = QuantizedLinearRowwiseTorch(layer_torch, device)
                
#                         tr0 = time.time()
#                         out_real = x @ layer
#                         tr1 = time.time()
#                         tq0 = time.time()
#                         out_quant = q_layer(x)
#                         tq1 = time.time()
#                         deallocate_tensors_from_gpu(layer_torch, q_layer, out_quant, out_real)

#                 stats['time_torch'].append(tr1-tr0)
#                 stats['time_quant'].append(tq1-tq0)
#                 for k, v in calc_quant_error_metrics(out_real, out_quant).items():
#                     stats[k].append(v)

#             print(f"seq_len: {N}, lin_shape: ({IN, OUT})")
#             print(f"\ttime_torch: {np.mean(stats['time_torch']):.6f} ; time_quant: {np.mean(stats['time_quant']):.6f}")
#             print(f"\trmse: {np.mean(stats['rmse']):.6f} ; mean_ape: {np.mean(stats['mean_ape']):.6f} ; median_ape: {np.mean(stats['median_ape']):.6f}")

In [38]:
# # int16 packing btw
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type='nf4',
# )

# gc.collect()
# torch.cuda.empty_cache()
# bench_matmul(seq_lens, linl_shapes, bnb_config=bnb_config)
# bench_matmul(seq_lens, linl_shapes, torch_global=True)
# bench_matmul(seq_lens, linl_shapes, torch_rowwise=True)

In [55]:
seq_lens = (128, 512, 2048)
linl_shapes = ((2048, 2048), (2048, 512), (2048, 8192), (2048, 128256))

In [26]:
device = 'cuda:0'

# int16 packing btw
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

torch.manual_seed(59)
for linls in linl_shapes:
    IN, OUT = linls
    layer = torch.empty(IN, OUT, dtype=torch.float32, requires_grad=False).to(device)
    nn.init.kaiming_uniform_(layer, mode='fan_in', nonlinearity='relu')
    q_layer, q_state = bbF.quantize_nf4(layer)
    check_mm(q_layer, q_state)

    for N in seq_lens:
        x = torch.randn(1, N, 2048).to(device)
        dq_layer = bbF.dequantize_nf4(q_layer, q_state)
        out_real = x @ layer
        out_quant = x @ dq_layer
        m = calc_quant_error_metrics(out_real, out_quant)
        print(f'W_shape: {linls}, seq_len: {N}, rmse: {m["rmse"]:.6f}, mean_ape: {m["mean_ape"]:.6f}, median_ape: {m["median_ape"]:.6f}')

deallocate_tensors_from_gpu(layer, q_layer, x, dq_layer, out_real, out_quant)
torch.cuda.empty_cache()

100%|██████████| 1000/1000 [00:30<00:00, 32.67it/s]


128: 157.904 us
512: 167.6 us
2048: 325.285 us
W_shape: (2048, 2048), seq_len: 128, rmse: 0.129188, mean_ape: 93.985001, median_ape: 9.119001
W_shape: (2048, 2048), seq_len: 512, rmse: 0.128882, mean_ape: 72.799454, median_ape: 9.077173
W_shape: (2048, 2048), seq_len: 2048, rmse: 0.129310, mean_ape: 82.364113, median_ape: 9.095244


100%|██████████| 1000/1000 [00:30<00:00, 33.20it/s]


128: 153.253 us
512: 172.097 us
2048: 334.85 us
W_shape: (2048, 512), seq_len: 128, rmse: 0.258943, mean_ape: 66.412582, median_ape: 9.187169
W_shape: (2048, 512), seq_len: 512, rmse: 0.258386, mean_ape: 61.140537, median_ape: 9.092073
W_shape: (2048, 512), seq_len: 2048, rmse: 0.258264, mean_ape: 76.330315, median_ape: 9.092836


100%|██████████| 1000/1000 [00:29<00:00, 33.51it/s]


128: 160.963 us
512: 157.657 us
2048: 276.797 us
W_shape: (2048, 8192), seq_len: 128, rmse: 0.064705, mean_ape: 74.763893, median_ape: 9.072249
W_shape: (2048, 8192), seq_len: 512, rmse: 0.064465, mean_ape: 74.898514, median_ape: 9.084351
W_shape: (2048, 8192), seq_len: 2048, rmse: 0.064601, mean_ape: 84.881935, median_ape: 9.089778


100%|██████████| 1000/1000 [02:11<00:00,  7.63it/s]


128: 154.04 us
512: 186.532 us
2048: 321.507 us
W_shape: (2048, 128256), seq_len: 128, rmse: 0.016321, mean_ape: 89.591408, median_ape: 9.093439
W_shape: (2048, 128256), seq_len: 512, rmse: 0.016317, mean_ape: 121.414444, median_ape: 9.089481
W_shape: (2048, 128256), seq_len: 2048, rmse: 0.016324, mean_ape: 131.609207, median_ape: 9.092866


In [56]:
device = 'cuda:0'
# torch global quant
torch.manual_seed(59)
for linls in linl_shapes:
    IN, OUT = linls
    layer = torch.empty(IN, OUT, dtype=torch.float32, requires_grad=False).to(device)
    nn.init.kaiming_uniform_(layer, mode='fan_in', nonlinearity='relu')

    layer_torch = nn.Linear(IN, OUT, bias=False, dtype=torch.float32).to(device)
    with torch.no_grad():
        layer_torch.weight.copy_(layer.T)
    q_layer = QuantizedLinearGlobalTorch(layer_torch, device)
    check_mm(q_layer)

    
    for N in seq_lens:
        x = torch.randn(1, N, 2048).to(device)
        out_real = layer_torch(x).detach().cpu()
        out_quant = q_layer(x).detach().cpu()
        m = calc_quant_error_metrics(out_real, out_quant)
        print(f'W_shape: {linls}, seq_len: {N}, rmse: {m["rmse"]:.6f}, mean_ape: {m["mean_ape"]:.6f}, median_ape: {m["median_ape"]:.6f}')

deallocate_tensors_from_gpu(layer, layer_torch, q_layer, x, out_real, out_quant)

100%|██████████| 1000/1000 [00:32<00:00, 30.78it/s]


128: 330.938 us
512: 335.748 us
2048: 518.394 us
W_shape: (2048, 2048), seq_len: 128, rmse: 0.100928, mean_ape: 74.974068, median_ape: 7.124235
W_shape: (2048, 2048), seq_len: 512, rmse: 0.101133, mean_ape: 58.410549, median_ape: 7.126679
W_shape: (2048, 2048), seq_len: 2048, rmse: 0.100989, mean_ape: 85.139191, median_ape: 7.126712


100%|██████████| 1000/1000 [00:30<00:00, 32.99it/s]


128: 339.319 us
512: 343.59 us
2048: 518.438 us
W_shape: (2048, 512), seq_len: 128, rmse: 0.202182, mean_ape: 64.189888, median_ape: 7.142402
W_shape: (2048, 512), seq_len: 512, rmse: 0.202135, mean_ape: 65.918480, median_ape: 7.149049
W_shape: (2048, 512), seq_len: 2048, rmse: 0.202168, mean_ape: 61.348331, median_ape: 7.135268


100%|██████████| 1000/1000 [00:30<00:00, 33.01it/s]


128: 337.915 us
512: 336.8 us
2048: 498.597 us
W_shape: (2048, 8192), seq_len: 128, rmse: 0.050493, mean_ape: 52.296341, median_ape: 7.114012
W_shape: (2048, 8192), seq_len: 512, rmse: 0.050498, mean_ape: 69.885567, median_ape: 7.127803
W_shape: (2048, 8192), seq_len: 2048, rmse: 0.050526, mean_ape: 85.378197, median_ape: 7.125717


100%|██████████| 1000/1000 [02:32<00:00,  6.54it/s]


128: 346.082 us
512: 384.326 us
2048: 559.503 us
W_shape: (2048, 128256), seq_len: 128, rmse: 0.012755, mean_ape: 81.647324, median_ape: 7.119849
W_shape: (2048, 128256), seq_len: 512, rmse: 0.012757, mean_ape: 77.641396, median_ape: 7.126124
W_shape: (2048, 128256), seq_len: 2048, rmse: 0.012769, mean_ape: 95.987679, median_ape: 7.123872


In [59]:
device = 'cuda:0'
# torch rowwise quant
torch.manual_seed(59)
for linls in linl_shapes:
    IN, OUT = linls
    layer = torch.empty(IN, OUT, dtype=torch.float32, requires_grad=False).to(device)
    nn.init.kaiming_uniform_(layer, mode='fan_in', nonlinearity='relu')

    layer_torch = nn.Linear(IN, OUT, bias=False, dtype=torch.float32).to(device)
    with torch.no_grad():
        layer_torch.weight.copy_(layer.T)
    q_layer = QuantizedLinearRowwiseTorch(layer_torch, device)
    check_mm(q_layer)

    
    for N in seq_lens:
        x = torch.randn(1, N, 2048).to(device)
        
        out_real = layer_torch(x).detach().cpu()
        out_quant = q_layer(x).detach().cpu()
        m = calc_quant_error_metrics(out_real, out_quant)
        print(f'W_shape: {linls}, seq_len: {N}, rmse: {m["rmse"]:.6f}, mean_ape: {m["mean_ape"]:.6f}, median_ape: {m["median_ape"]:.6f}')

    deallocate_tensors_from_gpu(layer, layer_torch, q_layer, x, out_real, out_quant)

100%|██████████| 1000/1000 [00:31<00:00, 32.12it/s]


128: 338.333 us
512: 335.904 us
2048: 518.672 us
W_shape: (2048, 2048), seq_len: 128, rmse: 0.100854, mean_ape: 70.818802, median_ape: 7.109579
W_shape: (2048, 2048), seq_len: 512, rmse: 0.101057, mean_ape: 58.528595, median_ape: 7.121001
W_shape: (2048, 2048), seq_len: 2048, rmse: 0.100905, mean_ape: 90.275932, median_ape: 7.119931


100%|██████████| 1000/1000 [00:30<00:00, 32.84it/s]


128: 335.934 us
512: 325.027 us
2048: 531.816 us
W_shape: (2048, 512), seq_len: 128, rmse: 0.202010, mean_ape: 66.787743, median_ape: 7.155147
W_shape: (2048, 512), seq_len: 512, rmse: 0.201949, mean_ape: 64.493820, median_ape: 7.120828
W_shape: (2048, 512), seq_len: 2048, rmse: 0.202028, mean_ape: 61.449711, median_ape: 7.130511


100%|██████████| 1000/1000 [00:30<00:00, 32.35it/s]


128: 332.185 us
512: 326.624 us
2048: 461.153 us
W_shape: (2048, 8192), seq_len: 128, rmse: 0.050460, mean_ape: 52.047253, median_ape: 7.107063
W_shape: (2048, 8192), seq_len: 512, rmse: 0.050458, mean_ape: 70.205818, median_ape: 7.122395
W_shape: (2048, 8192), seq_len: 2048, rmse: 0.050488, mean_ape: 85.376404, median_ape: 7.119174


100%|██████████| 1000/1000 [02:33<00:00,  6.54it/s]


128: 344.222 us
512: 385.251 us
2048: 555.041 us
W_shape: (2048, 128256), seq_len: 128, rmse: 0.012746, mean_ape: 83.166954, median_ape: 7.114754
W_shape: (2048, 128256), seq_len: 512, rmse: 0.012747, mean_ape: 75.714272, median_ape: 7.120488
W_shape: (2048, 128256), seq_len: 2048, rmse: 0.012760, mean_ape: 99.869217, median_ape: 7.119109


In [ ]:
# TODO: check perplexity for each quant method o_O

In [13]:
custom_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32, device_map='cuda:1')
custom_model = change_linear_layer(custom_model, QuantizedLinearGlobalTorch, custom_model.device)

In [31]:
text = 'Hello how is the weather?'
orig_model_size = calc_model_size(model)
orig_inf_time, orig_inf_text = time_inference(model, tokenizer, model.device, text)
print('%f GB; %f s.;\n%s' % (orig_model_size, orig_inf_time, orig_inf_text))

4.603768 GB; 1.075579 s.;
Hello how is the weather? I'm looking for a place to get some fresh air and enjoy the outdoors. I'm in the area of Los Angeles, California.

There are plenty of options for you to choose from, whether you're looking for


In [32]:
text = 'Hello how is the weather?'
quant_model_size = calc_model_size(custom_model)
quant_inf_time, quant_inf_text = time_inference(custom_model, tokenizer, custom_model.device, text)
print('%f GB; %f s.;\n%s' % (quant_model_size, quant_inf_time, quant_inf_text))

1.978768 GB; 8.088783 s.;
Hello how is the weather?Leading leading leading leading leading leadingleadleadLeadLead lead lead lead LeadLead lead lead lead lead lead lead lead lead lead Lead lead lead lead lead lead lead lead lead lead lead lead lead contact contact contact contact contact contact


In [33]:
verify_layer_accuracy(model, custom_model, tokenizer)

torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 0: mse = 0.000000, cosine_sim = 1.000000
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 1: mse = 0.005562, cosine_sim = 0.993758
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 2: mse = 9.103346, cosine_sim = 0.984845
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 3: mse = 9.147719, cosine_sim = 0.984810
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 4: mse = 9.176298, cosine_sim = 0.984829
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 5: mse = 9.219006, cosine_sim = 0.984871
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 6: mse = 9.254936, cosine_sim = 0.984943
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 7: mse = 9.293110, cosine_sim = 0.985044
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 8: mse = 9.339248, cosine_sim = 0.985086
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 9: mse = 9.376293, cosine_sim = 0.985096
torch.Size([1, 2, 2048]) torch